# CoffeeGuard AI — release evaluation

This notebook summarizes the committed evaluation of the deployed EfficientNetV2-B0 bundle and compares the candidate models using the same held-out test set. It reads only small JSON/CSV artifacts committed with the repository.

The test set contains **379 held-out photos**. Treat these metrics as benchmark results for this dataset, not as validation on new farm images.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
METRICS = ROOT / "artifacts" / "metrics"
EVAL = ROOT / "artifacts" / "eval"


In [ ]:
comparison = pd.DataFrame(json.loads((METRICS / "model_comparison.json").read_text()))
comparison["test_macro_f1_pct"] = comparison["test_macro_f1"] * 100
comparison[["bundle", "model", "test_macro_f1", "test_macro_f1_pct", "test_errors", "ece_after", "latency_model_p50_ms", "onnx_mb"]].sort_values("test_macro_f1", ascending=False)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ordered = comparison.sort_values("test_macro_f1", ascending=True)
axes[0].barh(ordered["bundle"], ordered["test_macro_f1"], color="#2d6a4f")
axes[0].set_title("Test macro-F1")
axes[0].set_xlim(0.90, 1.0)
axes[0].set_xlabel("Macro-F1")
axes[1].barh(ordered["bundle"], ordered["ece_after"], color="#e76f51")
axes[1].set_title("Calibration error after temperature scaling")
axes[1].set_xlabel("ECE (lower is better)")
axes[2].scatter(comparison["latency_model_p50_ms"], comparison["test_macro_f1"], s=comparison["onnx_mb"] * 20, alpha=0.8, color="#457b9d")
for _, row in comparison.iterrows():
    axes[2].annotate(row["bundle"], (row["latency_model_p50_ms"], row["test_macro_f1"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
axes[2].set_title("Accuracy / latency / model size")
axes[2].set_xlabel("CPU model p50 (ms); bubble ∝ ONNX MB")
axes[2].set_ylabel("Test macro-F1")
plt.tight_layout()
plt.show()


## Deployed bundle: confusion matrix and confidence behaviour


In [ ]:
pred_path = ROOT / "artifacts" / "app" / "predictions_test.csv"
pred = pd.read_csv(pred_path)
labels = ["Healthy", "Cercospora", "Leaf Rust", "Phoma"]
cm = pd.crosstab(pred["label"], pred["pred_label"]).reindex(index=labels, columns=labels, fill_value=0)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", cbar=False, ax=ax)
ax.set_title("Release bundle — test confusion matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.tight_layout()
plt.show()


In [ ]:
accuracy = pred["correct"].mean()
print(f"Rows: {len(pred):,}")
print(f"Accuracy: {accuracy:.3%}")
print(f"Macro-F1: {comparison.loc[comparison.bundle == 'cand-effv2b0-bgswap', 'test_macro_f1'].iloc[0]:.3%}")
print(f"Median confidence (correct): {pred.loc[pred.correct, 'confidence'].median():.3f}")
print(f"Median confidence (incorrect): {pred.loc[~pred.correct, 'confidence'].median():.3f}")


In [ ]:
# Confidence reliability: accuracy in confidence bins.
pred["confidence_bin"] = pd.cut(pred["confidence"], bins=np.linspace(0.5, 1.0, 11), include_lowest=True)
reliability = pred.groupby("confidence_bin", observed=False).agg(
    mean_confidence=("confidence", "mean"), accuracy=("correct", "mean"), count=("correct", "size")
).dropna()
reliability


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([0.5, 1.0], [0.5, 1.0], "--", color="gray", label="perfect calibration")
ax.plot(reliability["mean_confidence"], reliability["accuracy"], "o-", color="#2d6a4f", label="release bundle")
ax.set(xlim=(0.5, 1.0), ylim=(0.5, 1.0), xlabel="Mean confidence", ylabel="Observed accuracy", title="Reliability on the held-out test set")
ax.legend()
plt.tight_layout()
plt.show()


## Release decision

The selected bundle is `cand-effv2b0-bgswap` / `coffeeguard-effv2b0-v1`. Its key reported properties are **test macro-F1 0.974**, **95% bootstrap CI [0.956, 0.990]**, **ECE 0.014 after calibration**, and **15.5 ms median model latency** on the benchmark CPU. The bundle also includes a quality gate, KNN-based out-of-distribution gate, conformal prediction set, and refusal behaviour.

Do not interpret high held-out performance as proof of production reliability: the model card notes that it has not yet been validated on new farm photos.
